In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

print(torch.__version__)

2.13.0+cpu


In [8]:
# Database & DataLoaders
from torch.utils.data import DataLoader

import torchvision.transforms as transforms 

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

Build the CNN 

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flattening
        x = self.fc_layers(x)

        return x 


In [10]:
model = CNN()

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [6]:
epochs = 10 
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader = DataLoader(testset, batch_size=32, shuffle=False)
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)   # forward pass
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1} & loss={epoch_training_loss/len(trainloader)}")

epoch=1 & loss=1.3045922929632001
epoch=2 & loss=0.8751626093648446
epoch=3 & loss=0.7085861345170327
epoch=4 & loss=0.5875365941267477
epoch=5 & loss=0.47372081657002846
epoch=6 & loss=0.39043380284797513
epoch=7 & loss=0.307355963475454
epoch=8 & loss=0.24143274478561896
epoch=9 & loss=0.19569122366683459
epoch=10 & loss=0.16599401997155627


In [7]:
# Evaluate our CNN 

correct_labels = 0 
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.96000000000001
